# Engineering RAG — Full Benchmark Evaluation

**10 questions per benchmark × 6 benchmarks = 60 questions total**

| Dataset | Type | Questions | What it tests |
|---|---|---|---|
| SQuAD 2.0 | Fact extraction | 10 | Exact answer spans + unanswerable |
| HotpotQA | Multi-hop | 10 | Combining facts from 2 documents |
| Natural Questions | Open QA | 10 | Real user queries (informal language) |
| MS MARCO | Retrieval | 10 | NDCG@10, MRR@10 retrieval quality |
| RAGAS Synthetic | Hallucination | 10 | Faithfulness on our own documents |
| Custom (test_questions.py) | Domain-specific | 10 | Our 3 engineering PDFs (balanced) |

**Cost**: ~$0.08–0.12 | **Time**: ~10 minutes

## Step 1 — Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
os.chdir('..')

from pathlib import Path
from configs.settings import HAS_OPENAI, DATABASE_URL, EMBED_MODEL

print(f'OpenAI key : {"SET" if HAS_OPENAI else "MISSING — set OPENAI_API_KEY in .env"}')
print(f'Embed model: {EMBED_MODEL}')
print(f'Database   : {DATABASE_URL[:40]}...')

BENCHMARK_DIR = Path('data/benchmarks')
N = 10  # questions per benchmark
SEED = 42

## Step 2 — Prepare Benchmark Data Files

Creates sample files for any missing benchmark datasets.
To use **real** datasets, download them first:
- SQuAD: https://rajpurkar.github.io/SQuAD-explorer/ → `dev-v2.0.json`
- HotpotQA: https://hotpotqa.github.io/ → `hotpot_dev_distractor_v1.json`

In [ ]:
import subprocess
result = subprocess.run(
    [sys.executable, 'download_benchmarks.py', '--datasets', 'all'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

## Step 3 — Connect to VectorStore

In [ ]:
from src.ingest.vectorstore import VectorStore

vs = VectorStore()
vs.init_schema()
stats = vs.get_stats()
print(f'Documents  : {stats["documents"]}')
print(f'Total chunks: {stats["total_chunks"]}')
print(f'By type    : {stats["chunks_by_type"]}')

if stats['documents'] == 0:
    print('\nWARNING: No documents ingested yet!')
    print('Run: python ingest_docs.py data/')

## Step 4 — Load 10 Questions Per Benchmark

In [ ]:
from src.evaluation.benchmarks.squad import load_squad
from src.evaluation.benchmarks.hotpotqa import load_hotpotqa
from src.evaluation.benchmarks.natural_questions import load_natural_questions
from src.evaluation.benchmarks.msmarco import load_msmarco
from src.evaluation.benchmarks.ragas_synth import load_synthetic_testset, generate_synthetic_testset
from src.evaluation.benchmarks.custom_loader import load_custom_balanced

datasets = {}
skipped  = []

# ── SQuAD ──
try:
    datasets['squad'] = load_squad(BENCHMARK_DIR / 'squad', max_samples=N, seed=SEED)
    print(f'SQuAD        : {len(datasets["squad"])} questions loaded')
except FileNotFoundError as e:
    print(f'SQuAD        : SKIPPED — {e}')
    skipped.append('squad')

# ── HotpotQA ──
try:
    datasets['hotpotqa'] = load_hotpotqa(BENCHMARK_DIR / 'hotpotqa', max_samples=N, seed=SEED)
    print(f'HotpotQA     : {len(datasets["hotpotqa"])} questions loaded')
except FileNotFoundError as e:
    print(f'HotpotQA     : SKIPPED — {e}')
    skipped.append('hotpotqa')

# ── Natural Questions ──
try:
    datasets['natural_questions'] = load_natural_questions(BENCHMARK_DIR / 'natural_questions', max_samples=N, seed=SEED)
    print(f'Natural Qs   : {len(datasets["natural_questions"])} questions loaded')
except FileNotFoundError as e:
    print(f'Natural Qs   : SKIPPED — {e}')
    skipped.append('natural_questions')

# ── MS MARCO ──
try:
    msmarco_samples, msmarco_qrels = load_msmarco(BENCHMARK_DIR / 'msmarco', max_queries=N, seed=SEED)
    datasets['msmarco'] = msmarco_samples
    print(f'MS MARCO     : {len(datasets["msmarco"])} queries loaded')
except FileNotFoundError as e:
    print(f'MS MARCO     : SKIPPED — {e}')
    skipped.append('msmarco')
    msmarco_qrels = {}

# ── RAGAS Synthetic ──
synth_path = BENCHMARK_DIR / 'ragas_synth' / 'testset.json'
if synth_path.exists():
    datasets['ragas_synth'] = load_synthetic_testset(synth_path)[:N]
    print(f'RAGAS Synth  : {len(datasets["ragas_synth"])} questions loaded (cached)')
else:
    print('RAGAS Synth  : No cached testset — will generate from ingested docs')
    # Generated in Step 5 below

# ── Custom (test_questions.py — balanced 2 per category) ──
datasets['custom'] = load_custom_balanced(per_category=2, seed=SEED)
print(f'Custom       : {len(datasets["custom"])} questions loaded (2 per category × 5 categories)')

print(f'\nTotal loaded: {sum(len(v) for v in datasets.values())} questions across {len(datasets)} datasets')
if skipped:
    print(f'Skipped      : {skipped} (run download_benchmarks.py --real to get real data)')

## Step 5 — Generate RAGAS Synthetic (if not cached)

In [ ]:
if 'ragas_synth' not in datasets:
    print('Generating 10 synthetic questions from ingested documents...')
    print('This uses GPT-4o-mini — cost ~$0.02, time ~2 min')

    # Fetch real chunks from the vectorstore as source material
    sample_chunks = vs.search(
        vs.embed_query('engineering specifications maintenance procedure safety'),
        limit=50
    )

    synth_path.parent.mkdir(parents=True, exist_ok=True)
    datasets['ragas_synth'] = generate_synthetic_testset(
        source_chunks=sample_chunks,
        num_questions=N,
        output_path=synth_path,
    )
    print(f'RAGAS Synth  : {len(datasets["ragas_synth"])} questions generated and saved')
else:
    print('RAGAS Synth already loaded — skipping generation')

## Step 6 — Preview Questions

In [ ]:
print('=' * 80)
print('  BENCHMARK QUESTION PREVIEW (first 2 per dataset)')
print('=' * 80)

for name, samples in datasets.items():
    print(f'\n[{name.upper()}]')
    for i, s in enumerate(samples[:2]):
        cat = s.metadata.get('category', s.metadata.get('dataset', ''))
        print(f'  Q{i+1} [{cat}]: {s.question}')
        print(f'      GT: {s.ground_truth[:80]}...' if len(s.ground_truth) > 80 else f'      GT: {s.ground_truth}')

## Step 7 — Run Full RAG Pipeline on All Benchmarks

Each question goes through: **HyDE → pgvector search → RRF → CRAG → GPT-4o-mini → Self-RAG**

In [ ]:
from src.evaluation.benchmarks.runner import BenchmarkRunner, compare_reports

runner  = BenchmarkRunner(vectorstore=vs)
reports = []

total_q   = sum(len(v) for v in datasets.items())
print(f'Running {len(datasets)} benchmarks ({sum(len(v) for v in datasets.values())} questions total)')
print('Expected time: ~8-12 minutes | Cost: ~$0.08-0.12\n')

for name, samples in datasets.items():
    print(f'\n{'='*60}')
    print(f'  {name.upper()} ({len(samples)} questions)')
    print('='*60)

    # Custom dataset uses real ingested PDFs — no cleanup needed
    # Other datasets ingest their own context passages temporarily
    cleanup = (name != 'custom')

    report = runner.run(
        dataset_name=name,
        samples=samples,
        cleanup_after=cleanup,
        verbose=True,
    )
    reports.append(report)
    print(f'  Done: Judge={report.avg_judge_score:.2f}/5.0  F1={report.avg_f1:.1%}  Latency={report.avg_latency_sec:.2f}s')

## Step 8 — Results Comparison Table

In [ ]:
compare_reports(reports)

## Step 9 — Per-Question Detail

In [ ]:
import pandas as pd

rows = []
for report in reports:
    for r in (report.results or []):
        rows.append({
            'dataset':    report.dataset_name,
            'category':   r.sample.metadata.get('category', '-'),
            'question':   r.sample.question[:60],
            'answer':     r.answer[:80],
            'judge':      round(r.judge_score, 2),
            'f1':         round(r.f1_score, 3),
            'factuality': round(r.factuality, 2),
            'latency_s':  round(r.latency_sec, 2),
            'confidence': r.confidence,
            'self_rag':   r.sample.metadata.get('self_rag_status', '-'),
        })

df = pd.DataFrame(rows)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_rows', 100)
print(df.to_string(index=False))

## Step 10 — Save Results

In [ ]:
import json
from datetime import datetime

results_dir = Path('results')
results_dir.mkdir(exist_ok=True)

timestamp = datetime.now().strftime('%Y%m%d_%H%M')

# Save CSV
csv_path = results_dir / f'benchmark_{timestamp}.csv'
df.to_csv(csv_path, index=False)
print(f'CSV saved: {csv_path}')

# Save summary JSON
summary = []
for r in reports:
    summary.append({
        'dataset':        r.dataset_name,
        'n_samples':      r.num_samples,
        'avg_em':         r.avg_em,
        'avg_f1':         r.avg_f1,
        'avg_judge':      r.avg_judge_score,
        'avg_factuality': r.avg_factuality,
        'avg_latency_s':  r.avg_latency_sec,
        'p99_latency_s':  r.p99_latency_sec,
        'mrr':            r.mrr,
        'sla_latency':    r.sla_latency_ok,
        'sla_quality':    r.sla_quality_ok,
        'sla_factuality': r.sla_factuality_ok,
    })

json_path = results_dir / f'benchmark_summary_{timestamp}.json'
with open(json_path, 'w') as f:
    json.dump(summary, f, indent=2)
print(f'Summary JSON: {json_path}')

print('\nFinal SLA check:')
for r in reports:
    lat  = '✓' if r.sla_latency_ok    else '✗'
    qual = '✓' if r.sla_quality_ok    else '✗'
    fact = '✓' if r.sla_factuality_ok else '✗'
    print(f'  {r.dataset_name:<20} Latency:{lat}  Quality:{qual}  Factuality:{fact}')